In [3]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [5]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [55]:
control_key = "is_control"
condition_keys = "perturbation"
condition_rep_keys = "condition_embeddings"
donor_rep_keys = None
random_seed = 42
dataset_name = "Sciplex3_72h_hvg_chembert"
sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_state"
if_adata_ref = None
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"

In [56]:
filePath = './data/raw/Sciplex3_72h_hvg2000.h5ad'
adata = sc.read_h5ad(filePath)
print(adata)

AnnData object with n_obs × n_vars = 82110 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts'


In [57]:
adata.obs[control_key] = (adata.obs[condition_keys] == "control")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    80026
True      2084
Name: count, dtype: int64


In [58]:
#adata.obs[donor_rep_keys] = 7
if donor_rep_keys is not None:
    print(adata.obs[donor_rep_keys].value_counts())

## splitting

In [59]:
obs = adata.obs.copy()
obs["plate_well"] = obs["plate"].astype(str) + "_" + obs["well"].astype(str)

well_counts = (
    obs.groupby(["dose_value", "perturbation", "time","cell_line"], observed=True)["plate_well"]
    .nunique()
    .reset_index(name="n_wells")
)

control_wells = well_counts.loc[well_counts["dose_value"] == 0, "n_wells"].sum()
perturb_wells = well_counts.loc[well_counts["dose_value"] != 0, "n_wells"].unique()

assert len(perturb_wells) == 1, f"Non-control wells not identical: {perturb_wells}"

ratio = control_wells / perturb_wells[0]
print(ratio)


4.0


In [60]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.2
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1 / ratio
    print(condition_list)
else:
    # 按condition分割 zeroshot
    n_test = max(1, int(len(condition_list) * test_ratio))
    test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()
    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1 / ratio

['Dacinostat (LAQ824)', 'Divalproex Sodium', 'PHA-680632', 'Ellagic acid', 'Aurora A Inhibitor I', 'Tazemetostat (EPZ-6438)', 'Quisinostat (JNJ-26481585) 2HCl', 'Ruxolitinib (INCB018424)', 'Panobinostat (LBH589)']
['Alisertib (MLN8237)', 'Abexinostat (PCI-24781)', 'Hesperadin', 'PCI-34051', 'EED226', 'Barasertib (AZD1152-HQPA)', 'Tozasertib (VX-680, MK-0457)', 'Belinostat (PXD101)', 'Ofloxacin', 'SRT2104 (GSK2245840)', 'Danusertib (PHA-739358)', 'Tucidinostat (Chidamide)', 'PFI-1 (PF-6405761)', 'BMS-911543', 'Fasudil (HA-1077) HCl', 'Roxadustat (FG-4592)', '(+)-JQ1', 'AT9283', 'G007-LK', 'JNJ-7706621', 'Mocetinostat (MGCD0103)', 'MC1568', 'Resveratrol', 'Rucaparib (AG-014699,PF-01367338) phosphate', 'Selisistat (EX 527)', 'MLN8054', 'ZM 447439', 'ITSA-1 (ITSA1)', 'TMP195', 'Iniparib (BSI-201)', 'Pracinostat (SB939)', 'UNC1999', 'FLLL32', 'Azacitidine ', 'SRT3025 HCl', 'Decitabine', '2-Methoxyestradiol (2-MeOE2)', 'Bisindolylmaleimide IX (Ro 31-8220 Mesylate)']


In [61]:
del adata

## latent embedding

In [62]:
n_comps = 100
n_hidden = 1024
n_layers = 2
#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
condition_rep_dict = pd.read_pickle("./data/processed/drug_embeddings_sciplex3_chembert.pkl")
condition_rep_dict = condition_rep_dict["drug_to_embedding"]
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [63]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [64]:
adata_control

AnnData object with n_obs × n_vars = 2084 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'normalized_m'
    layers: 'counts'

In [69]:
adata_train.obs.groupby(["perturbation", "dose_value"]).size().unstack()

/tmp/ipykernel_3831192/2348386785.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_train.obs.groupby(["perturbation", "dose_value"]).size().unstack()


dose_value,10.0,100.0,1000.0,10000.0
perturbation,,,,
2-Methoxyestradiol (2-MeOE2),493,474,428,262
(+)-JQ1,596,513,508,527
AT9283,241,204,173,208
Abexinostat (PCI-24781),470,407,215,70
Alisertib (MLN8237),352,229,210,174
Azacitidine,519,568,576,421
BMS-911543,596,565,609,553
Barasertib (AZD1152-HQPA),286,211,200,203
Belinostat (PXD101),516,465,305,80


In [52]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    batch_key = "donor",
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep #+ "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

[5.8375463  5.354406   1.9501438  1.5780631  1.5067968  1.0937451
 1.246151   1.2264448  1.2074707  1.1618967  1.1368848  1.1183059
 1.0951333  1.0958426  1.0376511  1.0230557  0.9935378  0.96924514
 0.9534577  0.95324284 0.9435656  0.9391015  0.9303444  0.9141392
 0.91689277 0.9132645  0.8911708  0.89943624 0.87796587 0.88712186
 0.8755274  0.8752326  0.87857795 0.8583885  0.86134166 0.8629994
 0.8577325  0.8563357  0.83870614 0.84377694 0.8465473  0.8353571
 0.8387981  0.8225953  0.83248043 0.8316292  0.8291407  0.819454
 0.80664617 0.81238115 0.80766225 0.7935453  0.8238538  0.80756164
 0.81335145 0.7978379  0.79655975 0.7992161  0.7950008  0.7695524
 0.7867531  0.7786105  0.7831043  0.77231044 0.77731574 0.76621544
 0.77222437 0.7546557  0.76565444 0.7665757  0.76717293 0.75472695
 0.74795485 0.7507273  0.7488286  0.7502011  0.7469626  0.7408961
 0.73155206 0.74127126 0.7391501  0.7345726  0.73174345 0.7249045
 0.7206975  0.72405344 0.71398455 0.70669943 0.71316373 0.7162799
 0.704

In [53]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [54]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/Sciplex3_24h_hvg_chembert_42_0.2_True_X_pca_100_None
AnnData object with n_obs × n_vars = 15494 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'normalized_m', 'pca'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 529593 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()